**Environment Setup and Data Ingestion**

This section establishes the runtime environment by linking to the project's configuration module. It then ingests the raw "Ed Sheeran" dataset extracted during the EDA phase. I immediately filter the dataset to focus on a single target (Artist + Song + Region) to ensure we are building a clean, single-variable time series for our initial model.

In [2]:
import pandas as pd
import numpy as np
import sys
import os

# Link to the src module for project-wide configuration
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import config

# Load the raw dataset
input_path = config.PROJ_ROOT / "data" / "raw" / "ed_sheeran_charts.csv"
df = pd.read_csv(input_path)
df['date'] = pd.to_datetime(df['date'])

# Filter for the primary modeling target
# We isolate "Shape of You" in the "Global" region to create a univariate series
target_song = "Shape of You"
target_region = "Global"

df_model = df[
    (df['title'] == target_song) &
    (df['region'] == target_region)
].copy()

print(f"Data Ingestion Complete. Loaded {len(df_model)} rows for '{target_song}'.")

Config loaded. Pointing to raw data at: /Users/joelangstaff/Downloads
Data Ingestion Complete. Loaded 1860 rows for 'Shape of You'.


**Data Cleaning (Removing the "Viral 50" Duplicates)**

Real-world chart data often contains duplicates where a song appears on multiple charts (e.g., Top 200 and Viral 50) on the same date. A time-series model requires a strictly unique index (one value per time step). Here, I can resolve duplicates by prioritizing the entry with the highest stream count (typically the Top 200 entry) and ensuring the timeline is monotonic (sorted chronologically).

In [3]:
# Sort by date (ascending) and streams (descending)
df_model = df_model.sort_values(['date', 'streams'], ascending=[True, False])

# Deduplicate: Keep the first occurrence (highest streams) for each date
df_model = df_model.drop_duplicates(subset=['date'], keep='first')

# Re-sort by date to ensure chronological order for feature shifting
df_model = df_model.sort_values('date').reset_index(drop=True)

print(f"Integrity Check Passed. Active Row Count: {len(df_model)}")

Integrity Check Passed. Active Row Count: 1785


**Feature Construction (Lag Variables & Rolling Statistics)s**

This is the core of the feature engineering process. I transform the time series into a supervised learning problem using Autoregression.

- Target ($y$): I shift the streams backward by 1 day to create the label to be predicted (Next Day's Streams).
- Lag Features ($X$): I create columns representing past values (Yesterday, Last Week). This allows the model to learn serial correlations
- Rolling Statistics: I calculate a 7-day moving average to provide the model with a "smoothed" view of the recent trend, reducing the impact of daily noise.

In [4]:
# 1. Target Variable (y): The value I want to predict (Next Day's Streams)
df_model['target'] = df_model['streams'].shift(-1)

# 2. Lag 1 (x): Short-term memory (Yesterday's Streams)
df_model['lag_1'] = df_model['streams'].shift(1)

# 3. Lag 7 (x): Seasonal memory (Same day last week)
df_model['lag_7'] = df_model['streams'].shift(7)

# 4. Rolling Mean (x): The 7-day trend signal
df_model['rolling_mean_7'] = df_model['streams'].rolling(window=7).mean()

print("Autoregressive features created.")

Autoregressive features created.


**Temporal Feature Extraction**

Raw date objects are not directly usable by regression algorithms. I decompose the date into categorical integers to capture cyclical seasonality. Specifically, encoding the "Day of Week" allows the model to learn weekly patterns (e.g., the "Friday Release Spike" or "Sunday Slump"). I also perform final data cleaning to remove rows with NaN values resulting from the lag operations.

In [5]:
# Decompose Date into categorical features
df_model['day_of_week'] = df_model['date'].dt.dayofweek
df_model['month'] = df_model['date'].dt.month
df_model['is_weekend'] = df_model['day_of_week'].isin([5, 6]).astype(int)

# Remove rows with missing values (NaN) caused by shifting
# (e.g., The first 7 days lack a 'lag_7' and the last day lacks a 'target')
df_model = df_model.dropna()

print(f"Temporal features extracted. Final Dataset Shape: {df_model.shape}")

Temporal features extracted. Final Dataset Shape: (1768, 16)


**Pipeline Serialization**
The final step is to persist the transformed dataset to the disk. I save the file to the data/processed directory, which serves as the clean input for the modeling phase. This ensures reproducibility and separates the "Feature Engineering" logic from the "Model Training" logic.

In [6]:
# Define output path
output_path = config.PROJ_ROOT / "data" / "processed" / "training_data.csv"

# Ensure directory exists
output_path.parent.mkdir(parents=True, exist_ok=True)

# Serialize to CSV (excluding index)
df_model.to_csv(output_path, index=False)

print("="*30)
print(f"PIPELINE SUCCESS")
print(f"Training data saved to: {output_path}")
print("="*30)

PIPELINE SUCCESS
Training data saved to: /Users/joelangstaff/Library/Mobile Documents/com~apple~CloudDocs/Code/machine-learning/TOPICS/Time-Series Forecasting/Projects/spotify-forecasting/data/processed/training_data.csv
